In [1]:
from collections import OrderedDict
from typing import List, Tuple, Optional, Union
import copy, os, random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets.utils.logging import disable_progress_bar
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter


import flwr
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import Metrics, Context
from flwr.common.config import unflatten_dict
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg, FedXgbBagging
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr.server.client_proxy import ClientProxy
from flwr.common import (
    Code,
    FitRes,
    FitIns,
    EvaluateIns,
    EvaluateRes,
    Parameters,
    Scalar,
    Status,
)
from flwr.common.typing import UserConfig

import argparse
import pandas as pd
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from imblearn.over_sampling import RandomOverSampler

import tensorflow as tf

import xgboost as xgb

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# DEVICE = torch.device("cpu")
print(f"Training on {DEVICE}")
print(f"Flower {flwr.__version__} / PyTorch {torch.__version__}")
disable_progress_bar()

/home/ehsan/miniconda3/envs/flower/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-01-28 12:07:09.809673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738094829.822475  426711 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738094829.826393  426711 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-28 12:07:09.839568: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-cr

Training on cuda
Flower 1.14.0 / PyTorch 2.5.1


In [9]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "fed_avg_flower"
meta_args.model = "mlp"

# meta_args.model = "cnn"SO FAR GOOD WITHOUT NORMALIZATION 
# meta_args.round = 20
# meta_args.epoch_iterations = 20
# meta_args.local_lr = 0.001
# meta_args.batch_size = 100
# meta_args.decay_weight = 1.0
# meta_args.data_type = ""
meta_args.round = 80 # 50
meta_args.epoch_iterations = 20
meta_args.local_lr = 0.001
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.remove_labels = [17, 21, 25, 29]
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "../../ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "../../ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    # "wisconsin_hdd_ssd_merged": "../ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    # "wisconsin_ssd_delay_10ms_merged":"../ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",

    # "wisconsin_hdd_delay_10ms_merged":"../ds/v3/selected_cols_merged/wisconsin-220g2-hdd-delayed-10ms_merged_V3.csv",
    # "utah_ssd_merged": "../ds/v3/selected_cols_merged/utah-6525-25g-25Gbps_ssd_merged.csv",
    # "utah_ssd_delay_30ms_merged":"../ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-30ms_merged_V3.csv",
    # "utah_ssd_delay_10ms_merged":"../ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-10ms_merged_V3.csv",
    }

print(meta_args)

def set_global_seed(seed):
    """Set the seed for all random number generators."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    tf.random.set_seed(seed)

set_global_seed(meta_args.seed)

Namespace(gpu=0, seed=1, repeat=1, device=device(type='cuda', index=0), log_path='fed_avg_flower', model='mlp', round=80, epoch_iterations=20, local_lr=0.001, batch_size=150, decay_weight=1.0, data_type='', remove_labels=[17, 21, 25, 29], features=['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max', 'sender_nic_send_bytes', 'sender_nic_receive_bytes', 'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes'], filenames={'wisconsin_ssd_merged': '../../ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv', 'wisconsin_hdd_merged': '../../ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv'})


In [10]:
def process_and_prepare_dmatrix(args, remove_labels=None, features=None, filenames=None):
    if remove_labels is None:
        remove_labels = [17, 21, 25, 29]
    if features is None:
        features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
                    'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
    if filenames is None:
        filenames = {
            "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
            # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

            "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
            # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",
        }

    clients_data_loaders = {}
    client_test_loaders = {}
    combined_X_test, combined_y_test = [], []
    test_data_dict = {}

    for client_name, file_path in filenames.items():
        # Step 1: Load the dataset and Label encoding and scaling
        df = pd.read_csv(file_path)
        
        # Step 2: Remove specified labels
        for lbl in remove_labels:
            df = df.drop(df[df.label_value == lbl].index)
        
        # Normalize for transfer learning 
        # df = normalize_df(df)

        X = df.drop(columns="label_value")[features]
        y = df.label_value

        encoder = LabelEncoder()
        scaler = StandardScaler()
        
        y = encoder.fit_transform(y)
        # X = scaler.fit_transform(X)

        # Step 3: Split into train and test sets
        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=args.seed)
        
       
        # X_train = scaler.fit_transform(X_train)
        # X_test = scaler.transform(X_test)

        # Step 4: Apply oversampling to training data
        X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)

        X_train = X_train.to_numpy() if not isinstance(X_train, np.ndarray) else X_train
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_train = y_train.to_numpy() if not isinstance(y_train, np.ndarray) else y_train
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test

        clients_data_loaders[client_name] = xgb.DMatrix(X_train, label=y_train)

        # Combine test data for unified test dataset
        combined_X_test.append(X_test)
        combined_y_test.append(y_test)

        client_test_loaders[client_name] = xgb.DMatrix(X_test, label=y_test)

        
    # Combine all test data
    combined_X_test = np.vstack(combined_X_test)
    combined_y_test = np.hstack(combined_y_test)
    total_classes = len(np.unique(combined_y_test))

    global_test_loader = xgb.DMatrix(combined_X_test, label=combined_y_test)


    args.input_size = len(features)
    args.output_size = total_classes

    return clients_data_loaders, client_test_loaders, global_test_loader, total_classes, args 


In [11]:
def set_log_path(args):
    import datetime
    path =  './log/' + args.log_path+ '/'
    if not os.path.exists(path):
        os.makedirs(path)
    path_log = os.path.join(path)
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    return path_log + '_' + str(timestamp)

def summarize_dataset(dataset):
    print("=== Dataset Summary ===")
    # Dataset length
    dataset_size = dataset.num_row()
    print(f"Total samples: {dataset_size}")
    
    # Inspect the data
    data_shape = dataset.feature_names
    print(f"Data shape: {data_shape}")
    
    # Inspect the labels
    labels = dataset.get_label()
    labels_shape = labels.shape
    labels_type = labels.dtype
    print(f"Labels shape: {labels_shape}, Labels type: {labels_type}")
    
    print("===========================")


In [12]:
def load_datasets(partition_id: int, data_loaders, args):
    clients_data_loaders, client_test_loaders = data_loaders

    client_name = list(args.filenames.keys())[int(partition_id)]
    
    trainloader = clients_data_loaders[client_name]
    testloader = client_test_loaders[client_name]
    valloader = testloader

    return trainloader, valloader, testloader


def replace_keys(input_dict, match="-", target="_"):
    """Recursively replace match string with target string in dictionary keys."""
    new_dict = {}
    for key, value in input_dict.items():
        new_key = key.replace(match, target)
        if isinstance(value, dict):
            new_dict[new_key] = replace_keys(value, match, target)
        else:
            new_dict[new_key] = value
    return new_dict

# load_datasets(2)

In [17]:
# def set_parameters(net, parameters: List[np.ndarray]):
#     params_dict = zip(net.state_dict().keys(), parameters)
#     state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
#     net.load_state_dict(state_dict, strict=True)


# def get_parameters(net) -> List[np.ndarray]:
#     return [val.cpu().numpy() for _, val in net.state_dict().items()]

In [13]:
class FlowerClient(Client):
    def __init__(self, train_dmatrix, valid_dmatrix, num_train, num_val, 
                    num_local_round, params, partition_id):
        self.train_dmatrix = train_dmatrix
        self.valid_dmatrix = valid_dmatrix
        self.num_train = num_train
        self.num_val = num_val
        self.num_local_round = num_local_round
        self.params = params
        self.p_id = partition_id
    def fit(self, ins: FitIns) -> FitRes:
        global_round = int(ins.config["global_round"])
        if global_round == 1:
            # First round local training
            bst = xgb.train(
                self.params,
                self.train_dmatrix,
                num_boost_round=self.num_local_round,
                evals=[(self.valid_dmatrix, "validate"), (self.train_dmatrix, "train")],
            )
        else:
            bst = xgb.Booster(params=self.params)
            global_model = bytearray(ins.parameters.tensors[0])

            # Load global model into booster
            bst.load_model(global_model)

            # Local training
            bst = self._local_boost(bst)

        # Save model
        local_model = bst.save_raw("json")
        local_model_bytes = bytes(local_model)

        return FitRes(
            status=Status(
                code=Code.OK,
                message="OK",
            ),
            parameters=Parameters(tensor_type="", tensors=[local_model_bytes]),
            num_examples=self.num_train,
            metrics={},
        )

    def evaluate(self, ins: EvaluateIns) -> EvaluateRes:
        # Load global model
        bst = xgb.Booster(params=self.params)
        para_b = bytearray(ins.parameters.tensors[0])
        bst.load_model(para_b)

        # Run evaluation
        eval_results = bst.eval_set(
            evals=[(self.valid_dmatrix, "valid")],
            iteration=bst.num_boosted_rounds() - 1,
        )
        # auc = round(float(eval_results.split("\t")[1].split(":")[1]), 4)
        
        # eval_result might look like: "[round]\tvalid-mlogloss:0.xxx"
        # parse out the mlogloss
        items = eval_results.split("\t")
        # We expect something like ["[0]", "valid-mlogloss:0.5"]
        mlogloss_value = 0.0
        for it in items:
            if "valid-mlogloss:" in it:
                mlogloss_value = float(it.split(":")[1])
                break

        return EvaluateRes(
            status=Status(
                code=Code.OK,
                message="OK",
            ),
            loss=mlogloss_value,
            num_examples=self.num_val,
            metrics={"mlogloss": mlogloss_value},
        )
        


def create_client(args, data_loaders, run_config: UserConfig) -> ClientApp:
    
    def client_fn(context: Context) -> Client:
        # Load model and data
        partition_id = context.node_config["partition-id"]
        # num_partitions = context.node_config["num-partitions"]
        # train_dmatrix, valid_dmatrix, num_train, num_val = load_datasets(
        #     partition_id, num_partitions
        # )
        train_dmatrix, valid_dmatrix, test_dmatrix = load_datasets(partition_id=partition_id, 
                                                                   data_loaders=data_loaders,
                                                                   args=args)

        cfg = replace_keys(unflatten_dict(run_config))
        num_local_round = cfg["local_epochs"]

        # Return Client instance
        return FlowerClient(
            train_dmatrix,
            valid_dmatrix,
            train_dmatrix.num_row(),
            valid_dmatrix.num_row(),
            num_local_round,
            cfg["params"],
        )
    # Create the ClientApp
    client = ClientApp(client_fn=client_fn)

    return client

In [16]:
def evaluate_metrics_aggregation(eval_metrics):
    """Return an aggregated metric (mlogloss) for evaluation."""
    total_num = sum([num for num, _ in eval_metrics])
    # Weighted average of mlogloss by number of examples
    mlogloss_aggregated = sum([
        metrics["mlogloss"] * num for num, metrics in eval_metrics
    ]) / total_num if total_num > 0 else 0.0

    metrics_aggregated = {"mlogloss": mlogloss_aggregated}
    return metrics_aggregated


def config_func(rnd: int) -> dict[str, str]:
    """Return a configuration with global epochs."""
    config = {
        "global_round": str(rnd),
    }
    return config


def create_server(global_test_loader, args, run_config):


    def server_fn(context: Context):
        # Read from config
        num_rounds = run_config["num-server-rounds"]
        fraction_fit = run_config["fraction-fit"]
        fraction_evaluate = run_config["fraction-evaluate"]

        # Optionally set initial global model (here, empty)
        parameters = Parameters(tensor_type="", tensors=[])

        # Define strategy
        strategy = FedXgbBagging(
            fraction_fit=fraction_fit,
            fraction_evaluate=fraction_evaluate,
            evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation,
            on_evaluate_config_fn=config_func,
            on_fit_config_fn=config_func,
            initial_parameters=parameters,
        )
        config = ServerConfig(num_rounds=num_rounds)

        return ServerAppComponents(strategy=strategy, config=config)

    # Build the ServerApp
    server_app = ServerApp(server_fn=server_fn)

    return server_app

In [17]:
args = copy.deepcopy(meta_args)
clients_data_loaders, client_test_loaders, global_test_loader, total_classes, args = process_and_prepare_dmatrix(args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)

# print(clients_data_loaders, "\n")
# print(client_test_loaders, "\n")
summarize_dataset(client_test_loaders["wisconsin_ssd_merged"])


trainloader, valloader, testloader = load_datasets(partition_id=0, data_loaders=(clients_data_loaders, client_test_loaders), args=args)

client_run_config = {
  "local-epochs": 1,
  "params": {
    "objective": "multi:softprob",
    # "objective": "multi:softmax",
    "num_class": args.output_size,
    "eta": 0.1,
    "max-depth": 8,
    "eval_metric": "mlogloss",
    # "eval_metric": "auc",
    "nthread": 16,
    "num-parallel-tree": 1,
    "subsample": 1,
    "tree-method": "hist"
  }
}

client = create_client(args=args, data_loaders=(clients_data_loaders, client_test_loaders), run_config=client_run_config)


server_run_config = {
    "num-server-rounds": 3,
    "fraction-fit": 0.1,
    "fraction-evaluate": 0.1
}

server = create_server(global_test_loader=global_test_loader, args=args, run_config=server_run_config)


run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=2,
    backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}
)


INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)


=== Dataset Summary ===
Total samples: 1408
Data shape: None
Labels shape: (1408,), Labels type: float32


ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     Traceback (most recent call last):
  File "/home/ehsan/miniconda3/envs/flower/lib/python3.12/site-packages/flwr/server/superlink/fleet/vce/backend/raybackend.py", line 166, in process_message
    future = self.pool.submit(
             ^^^^^^^^^^^^^^^^^
  File "/home/ehsan/miniconda3/envs/flower/lib/python3.12/site-packages/flwr/simulation/ray_transport/ray_actor.py", line 461, in submit
    future = actor_fn(actor, app_fn, mssg, cid, context)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ehsan/miniconda3/envs/flower/lib/python3.12/site-packages/flwr/server/superlink/fleet/vce/backend/raybackend.py", line 167, in <lambda>
    lambda a, a_fn, mssg, cid, state: a.run.remote(a_fn, mssg, cid, state),
               